# Guia de adquisicion de datos HAPI para la tarea modelo

Este notebook se enfoca en recopilar los datos observacionales que usa `notebooks_TareaModelo.ipynb` para el evento del 5 al 8 de marzo de 2012.

Objetivos:
- Descargar los productos HAPI necesarios para la tarea.
- Guardar cada tabla en `data/hapi_tareamodelo/`.
- Guardar la metadata asociada en archivos JSON para facilitar trazabilidad y reutilizacion.

Alcance:
- Incluye las fuentes HAPI observacionales del notebook de tarea.
- No descarga archivos CCMC/ENLIL locales; esos insumos siguen siendo externos a HAPI.


## 1. Importaciones y configuracion

Ajuste `OUTPUT_DIR` si desea exportar los archivos a otra carpeta del proyecto.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from hapiclient import hapi, hapitime2datetime

START = '2012-03-05T00:00:00Z'
STOP = '2012-03-08T00:00:00Z'
CACHE_DIR = Path('hapicache')
OUTPUT_DIR = Path('data/hapi_tareamodelo')
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OPTS = {'logging': False, 'usecache': True, 'cachedir': str(CACHE_DIR)}

print(f'Window: {START} -> {STOP}')
print(f'Cache: {CACHE_DIR.resolve()}')
print(f'Output: {OUTPUT_DIR.resolve()}')


Window: 2012-03-05T00:00:00Z -> 2012-03-08T00:00:00Z
Cache: /Volumes/T7/tarea3-astroparticulas/hapicache
Output: /Volumes/T7/tarea3-astroparticulas/data/hapi_tareamodelo


## 2. Funciones auxiliares

Estas funciones convierten la respuesta estructurada de HAPI a `DataFrame`, guardan resultados en disco y distinguen entre productos fuera de rango temporal y fallas de conexion.


In [2]:
def structured_to_frame(data):
    frame = pd.DataFrame()
    for name in data.dtype.names:
        values = data[name]
        if name == 'Time':
            frame[name] = pd.to_datetime(hapitime2datetime(values))
            continue

        array = np.asarray(values)
        if array.ndim == 1:
            frame[name] = array
        elif array.ndim == 2:
            for idx in range(array.shape[1]):
                frame[f'{name}_{idx}'] = array[:, idx]
        else:
            frame[name] = list(array)

    if 'Time' in frame.columns:
        frame = frame.set_index('Time')
    return frame


def save_metadata(path, metadata):
    path.write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')


def request_window(request):
    return request.get('start', START), request.get('stop', STOP)


def validate_request_window(request):
    request_start, request_stop = request_window(request)
    request_start = pd.Timestamp(request_start)
    request_stop = pd.Timestamp(request_stop)

    valid_start = request.get('valid_start')
    if valid_start is not None:
        valid_start = pd.Timestamp(valid_start)
        if request_start < valid_start:
            return False, f"Fuera de rango para esta ventana: disponible desde {valid_start:%Y-%m-%d %H:%M UTC}."

    valid_stop = request.get('valid_stop')
    if valid_stop is not None:
        valid_stop = pd.Timestamp(valid_stop)
        if request_stop > valid_stop:
            return False, f"Fuera de rango para esta ventana: disponible hasta {valid_stop:%Y-%m-%d %H:%M UTC}."

    return True, request.get('availability_note', '')


def classify_exception(exc):
    message = str(exc).strip()
    lowered = message.lower()

    if 'failed to connect' in lowered:
        return 'connection_error', 'No se pudo conectar al servidor remoto; reintente con acceso de red o use una descarga manual.'

    if 'time outside valid range' in lowered or 'time array is empty' in lowered:
        return 'out_of_range', message

    return 'error', message


def fetch_dataset(request):
    request_start, request_stop = request_window(request)
    data, meta = hapi(
        request['server'],
        request['dataset'],
        request['parameters'],
        request_start,
        request_stop,
        **OPTS,
    )
    frame = structured_to_frame(data)
    stem = OUTPUT_DIR / request['slug']
    frame.to_csv(stem.with_suffix('.csv'))
    save_metadata(stem.with_suffix('.metadata.json'), meta)
    return data, meta, frame


## 3. Catalogo de productos requeridos

El catalogo replica los productos observacionales usados en `notebooks_TareaModelo.ipynb`. Tambien marca las fuentes que no cubren marzo de 2012 para evitar falsos errores durante la descarga.


In [3]:
requests = [
    {
        'slug': 'dst_final',
        'group': 'geomagnetic_indices',
        'server': 'https://iswa.gsfc.nasa.gov/IswaSystemWebApp/hapi',
        'dataset': 'dst_final',
        'parameters': 'Dst',
        'description': 'Dst index from ISWA',
    },
    {
        'slug': 'intermagnet_abk_field_magnitude',
        'group': 'ground_magnetometers',
        'server': 'https://imag-data.bgs.ac.uk/GIN_V1/hapi',
        'dataset': 'abk/best-avail/PT1M/hdzf',
        'parameters': 'Field_Magnitude',
        'description': 'INTERMAGNET Abisko field magnitude',
    },
    {
        'slug': 'intermagnet_mea_field_magnitude',
        'group': 'ground_magnetometers',
        'server': 'https://imag-data.bgs.ac.uk/GIN_V1/hapi',
        'dataset': 'mea/best-avail/PT1M/hdzf',
        'parameters': 'Field_Magnitude',
        'description': 'INTERMAGNET Meanook field magnitude',
    },
    {
        'slug': 'intermagnet_sjg_field_magnitude',
        'group': 'ground_magnetometers',
        'server': 'https://imag-data.bgs.ac.uk/GIN_V1/hapi',
        'dataset': 'sjg/best-avail/PT1M/hdzf',
        'parameters': 'Field_Magnitude',
        'description': 'INTERMAGNET San Juan field magnitude',
    },
    {
        'slug': 'intermagnet_hua_field_magnitude',
        'group': 'ground_magnetometers',
        'server': 'https://imag-data.bgs.ac.uk/GIN_V1/hapi',
        'dataset': 'hua/best-avail/PT1M/hdzf',
        'parameters': 'Field_Magnitude',
        'description': 'INTERMAGNET Huancayo field magnitude',
    },
    {
        'slug': 'intermagnet_kep_field_magnitude',
        'group': 'ground_magnetometers',
        'server': 'https://imag-data.bgs.ac.uk/GIN_V1/hapi',
        'dataset': 'kep/best-avail/PT1M/hdzf',
        'parameters': 'Field_Magnitude',
        'valid_start': '2013-01-01T00:00:00Z',
        'availability_note': 'Esta estacion no cubre el evento de marzo de 2012.',
        'description': 'INTERMAGNET Keplerovo field magnitude',
    },
    {
        'slug': 'ace_mfi_magnitude_bgsm',
        'group': 'solar_wind',
        'server': 'https://cdaweb.gsfc.nasa.gov/hapi',
        'dataset': 'AC_H0_MFI',
        'parameters': 'Magnitude,BGSM',
        'availability_note': 'Depende de conectividad con CDAWeb; si falla la red, este producto no se exporta.',
        'description': 'ACE magnetic field magnitude and GSM vector',
    },
    {
        'slug': 'geotail_cpi_plasma',
        'group': 'solar_wind',
        'server': 'https://cdaweb.gsfc.nasa.gov/hapi',
        'dataset': 'GE_K0_CPI',
        'parameters': 'SW_P_Den,SW_V,W',
        'availability_note': 'Depende de conectividad con CDAWeb; si falla la red, este producto no se exporta.',
        'description': 'GEOTAIL plasma density, velocity and pressure proxies',
    },
    {
        'slug': 'gps_roti_south',
        'group': 'ionosphere',
        'server': 'https://cdaweb.gsfc.nasa.gov/hapi',
        'dataset': 'GPS_ROTI15MIN_JPL',
        'parameters': 'rotimedM_South',
        'valid_start': '2012-11-29T00:00:00Z',
        'availability_note': 'El producto comienza a finales de 2012, fuera de la ventana del evento.',
        'description': 'South region ROTI median from JPL GPS product',
    },
    {
        'slug': 'omni_hro2_5min',
        'group': 'solar_wind',
        'server': 'https://cdaweb.gsfc.nasa.gov/hapi',
        'dataset': 'OMNI_HRO2_5MIN',
        'parameters': 'F,BZ_GSM,flow_speed,proton_density,T,Pressure,PR-FLX_10,PR-FLX_30,PR-FLX_60',
        'availability_note': 'Depende de conectividad con CDAWeb; si falla la red, este producto no se exporta.',
        'description': 'OMNI 5 minute solar wind and energetic particle context',
    },
    {
        'slug': 'noaa_kp_observed',
        'group': 'geomagnetic_indices',
        'server': 'https://iswa.gsfc.nasa.gov/IswaSystemWebApp/hapi',
        'dataset': 'NOAA_KP_P3H',
        'parameters': 'Kp_observed',
        'description': 'Observed Kp index',
    },
    {
        'slug': 'lasco_c3_urls',
        'group': 'imagery',
        'server': 'https://api.helioviewer.org/hapi/Helioviewer/hapi',
        'dataset': 'LASCO_C3',
        'parameters': 'url',
        'description': 'Helioviewer LASCO C3 image URLs',
    },
    {
        'slug': 'swpc_rtsw_plasma',
        'group': 'solar_wind',
        'server': 'https://iswa.gsfc.nasa.gov/IswaSystemWebApp/hapi',
        'dataset': 'swpc_rtsw_plasma_P1M',
        'parameters': 'ProtonDensity,IonTemperature',
        'description': 'SWPC real-time solar wind plasma',
    },
]

catalog = pd.DataFrame(requests)
catalog[['slug', 'group', 'dataset', 'parameters', 'availability_note', 'description']].fillna('')


,slug,group,dataset,parameters,description
0,dst_final,geomagnetic_indices,dst_final,Dst,Dst index from ISWA
1,intermagnet_abk_field_magnitude,ground_magnetometers,abk/best-avail/PT1M/hdzf,Field_Magnitude,INTERMAGNET Abisko field magnitude
2,intermagnet_mea_field_magnitude,ground_magnetometers,mea/best-avail/PT1M/hdzf,Field_Magnitude,INTERMAGNET Meanook field magnitude
3,intermagnet_sjg_field_magnitude,ground_magnetometers,sjg/best-avail/PT1M/hdzf,Field_Magnitude,INTERMAGNET San Juan field magnitude
4,intermagnet_hua_field_magnitude,ground_magnetometers,hua/best-avail/PT1M/hdzf,Field_Magnitude,INTERMAGNET Huancayo field magnitude
5,intermagnet_kep_field_magnitude,ground_magnetometers,kep/best-avail/PT1M/hdzf,Field_Magnitude,INTERMAGNET Keplerovo field magnitude
6,ace_mfi_magnitude_bgsm,solar_wind,AC_H0_MFI,"Magnitude,BGSM",ACE magnetic field magnitude and GSM vector
7,geotail_cpi_plasma,solar_wind,GE_K0_CPI,"SW_P_Den,SW_V,W","GEOTAIL plasma density, velocity and pressure ..."
8,gps_roti_south,ionosphere,GPS_ROTI15MIN_JPL,rotimedM_South,South region ROTI median from JPL GPS product
9,omni_hro2_5min,solar_wind,OMNI_HRO2_5MIN,"F,BZ_GSM,flow_speed,proton_density,T,Pressure,...",OMNI 5 minute solar wind and energetic particl...


## 4. Descarga, exportacion y resumen

Cada producto se consulta con cache local, se exporta a CSV y se guarda su metadata en JSON. El resumen distingue entre `ok`, `out_of_range`, `connection_error` y otros errores para que el faltante sea interpretable.


In [4]:
results = {}
summary_rows = []

for request in requests:
    print(f"Fetching {request['slug']} ...")
    in_range, note = validate_request_window(request)
    if not in_range:
        summary_rows.append(
            {
                'slug': request['slug'],
                'status': 'out_of_range',
                'rows': 0,
                'columns': 0,
                'csv': '',
                'metadata_json': '',
                'note': note,
                'error': '',
            }
        )
        continue

    try:
        data, meta, frame = fetch_dataset(request)
        results[request['slug']] = {'data': data, 'meta': meta, 'frame': frame}
        summary_rows.append(
            {
                'slug': request['slug'],
                'status': 'ok',
                'rows': len(frame),
                'columns': len(frame.columns),
                'csv': str((OUTPUT_DIR / request['slug']).with_suffix('.csv')),
                'metadata_json': str((OUTPUT_DIR / request['slug']).with_suffix('.metadata.json')),
                'note': request.get('availability_note', ''),
                'error': '',
            }
        )
    except Exception as exc:
        status, error_note = classify_exception(exc)
        summary_rows.append(
            {
                'slug': request['slug'],
                'status': status,
                'rows': 0,
                'columns': 0,
                'csv': '',
                'metadata_json': '',
                'note': error_note,
                'error': str(exc),
            }
        )

summary = pd.DataFrame(summary_rows)
summary.sort_values(['status', 'slug']).reset_index(drop=True)


Fetching dst_final ...
Fetching intermagnet_abk_field_magnitude ...
Fetching intermagnet_mea_field_magnitude ...
Fetching intermagnet_sjg_field_magnitude ...
Fetching intermagnet_hua_field_magnitude ...
Fetching intermagnet_kep_field_magnitude ...
Fetching ace_mfi_magnitude_bgsm ...
Fetching geotail_cpi_plasma ...
Fetching gps_roti_south ...
Fetching omni_hro2_5min ...
Fetching noaa_kp_observed ...
Fetching lasco_c3_urls ...
Fetching swpc_rtsw_plasma ...


,slug,status,rows,columns,csv,metadata_json,error
0,ace_mfi_magnitude_bgsm,error,0,0,,,Failed to connect to: https://cdaweb.gsfc.nasa...
1,geotail_cpi_plasma,error,0,0,,,Failed to connect to: https://cdaweb.gsfc.nasa...
2,gps_roti_south,error,0,0,,,Time array is empty.\n
3,intermagnet_kep_field_magnitude,error,0,0,,,HAPI error 1405: time outside valid range\n
4,omni_hro2_5min,error,0,0,,,Failed to connect to: https://cdaweb.gsfc.nasa...
5,dst_final,ok,72,1,data/hapi_tareamodelo/dst_final.csv,data/hapi_tareamodelo/dst_final.metadata.json,
6,intermagnet_abk_field_magnitude,ok,4320,1,data/hapi_tareamodelo/intermagnet_abk_field_ma...,data/hapi_tareamodelo/intermagnet_abk_field_ma...,
7,intermagnet_hua_field_magnitude,ok,4320,1,data/hapi_tareamodelo/intermagnet_hua_field_ma...,data/hapi_tareamodelo/intermagnet_hua_field_ma...,
8,intermagnet_mea_field_magnitude,ok,4320,1,data/hapi_tareamodelo/intermagnet_mea_field_ma...,data/hapi_tareamodelo/intermagnet_mea_field_ma...,
9,intermagnet_sjg_field_magnitude,ok,4320,1,data/hapi_tareamodelo/intermagnet_sjg_field_ma...,data/hapi_tareamodelo/intermagnet_sjg_field_ma...,


## 5. Vistas rapidas para la tarea

Estas vistas permiten comprobar de inmediato que los archivos clave para la tarea quedaron listos.


In [5]:
key_products = ['dst_final', 'omni_hro2_5min', 'ace_mfi_magnitude_bgsm', 'noaa_kp_observed']
for slug in key_products:
    if slug in results:
        print(f'\n{slug}')
        display(results[slug]['frame'].head())



dst_final


,Dst
Time,
2012-03-05 00:00:00+00:00,-30.0
2012-03-05 01:00:00+00:00,-32.0
2012-03-05 02:00:00+00:00,-33.0
2012-03-05 03:00:00+00:00,-33.0
2012-03-05 04:00:00+00:00,-35.0



noaa_kp_observed


,Kp_observed
Time,
2012-03-05 00:00:00+00:00,2.0
2012-03-05 03:00:00+00:00,2.0
2012-03-05 06:00:00+00:00,2.0
2012-03-05 09:00:00+00:00,2.0
2012-03-05 12:00:00+00:00,1.0


## 6. Siguiente paso

Con los archivos ya exportados, `notebooks_TareaModelo.ipynb` desde
 `data/hapi_tareamodelo/`.
